# Word2Vec Practical

**Goal:** read local text files, preprocess them into tokenized sentences, then train custom CBOW and Skip-Gram Word2Vec models.

**Flow:** Read text files → sentence tokenize → `simple_preprocess()` cleanup → build vocabulary → train embeddings → inspect vectors and similarities

In [1]:
import numpy as np
import pandas as pd
import gensim
import nltk
import os
print("Libraries imported successfully!")

Libraries imported successfully!


## Step 1: Import libraries

- `os` is used to iterate through all files inside `./data/`.
- `nltk` provides sentence tokenization.
- `simple_preprocess` is a fast gensim utility for lowercasing and punctuation cleanup.
- `Word2Vec` trains dense word embeddings.

**Alternative:** manual tokenization with regex or `word_tokenize()`.

**Tradeoff:** `simple_preprocess()` is compact and clean for Word2Vec, but it gives less manual control than a fully custom preprocessing pipeline.

In [2]:
_ = nltk.download('all', quiet=True)
print("NLTK resources downloaded successfully!")

NLTK resources downloaded successfully!


In [3]:
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec
print("Additional libraries imported successfully!")

Additional libraries imported successfully!


## Step 2: Read files and preprocess text

- Each file is opened from `./data/`.
- `sent_tokenize()` splits long text into sentence-level units.
- `simple_preprocess(sentence)` converts each sentence into a clean token list.
- Final training input becomes `list[list[str]]`, which is the format gensim expects.

**Why sentence-level input?** Word2Vec learns from local context windows, so sentence boundaries matter.

**Alternative:** pass one large token stream, but sentence-wise tokenization gives cleaner context windows.

**Tradeoff:** this notebook keeps preprocessing simple; more advanced cleaning could improve results but adds complexity.

In [5]:
data_folder= './data/'
story_data =[]
for filename in os.listdir(data_folder):
    with open(os.path.join(data_folder, filename), "r", encoding="utf-8") as f:
        print(f)
        corpus = f.read()
    raw_sentences= sent_tokenize(corpus)
    for sentence in raw_sentences: 
        story_data.append(simple_preprocess(sentence))


<_io.TextIOWrapper name='./data/story2_data.txt' mode='r' encoding='utf-8'>
<_io.TextIOWrapper name='./data/word2vec_data.txt' mode='r' encoding='utf-8'>


In [6]:
print(story_data[:1])
print("\nTotal sentences:", len(story_data))
print('\nTotal tokenized words in sentences:', sum(len(sentence) for sentence in story_data))


[['jensen', 'huang', 'is', 'taiwanese', 'american', 'entrepreneur', 'engineer', 'and', 'business', 'leader', 'best', 'known', 'as', 'the', 'co', 'founder', 'and', 'long', 'serving', 'chief', 'executive', 'officer', 'of', 'nvidia', 'one', 'of', 'the', 'most', 'influential', 'technology', 'companies', 'in', 'the', 'world', 'particularly', 'in', 'the', 'fields', 'of', 'graphics', 'processing', 'artificial', 'intelligence', 'and', 'high', 'performance', 'computing']]

Total sentences: 165

Total tokenized words in sentences: 2141


## Step 3: Inspect the processed corpus

- Preview the first tokenized sentence.
- Check total sentence count.
- Check total token count across the corpus.

This verifies that preprocessing worked before training starts.

In [7]:
story_data[11]

['these',
 'roles',
 'helped',
 'him',
 'understand',
 'the',
 'industry',
 'deeply',
 'and',
 'prepared',
 'him',
 'for',
 'the',
 'challenges',
 'of',
 'building',
 'company',
 'from',
 'scratch']

In [8]:
cbow_model= Word2Vec(
    window=6,
    min_count=3,
    vector_size=300,
    epochs= 50,
    sg=0
)
print('hyperparameters set successfully!')

hyperparameters set successfully!


## Step 4: Configure and train the CBOW model

- `sg=0` means CBOW.
- `window=6` controls how many nearby words are treated as context.
- `min_count=3` removes very rare words.
- `vector_size=300` sets embedding dimension.
- `epochs=50` increases training passes because the corpus is small.

**Why separate `build_vocab()` and `train()`?** It makes the training pipeline explicit for revision.

**Alternative:** `Word2Vec(sentences=story_data, ...)` can build vocabulary automatically.

**Tradeoff:** explicit steps are better for learning; direct training is shorter but hides internal stages.

In [9]:
cbow_model.build_vocab(story_data)
print("Vocabulary built successfully!")

Vocabulary built successfully!


In [10]:
cbow_model.corpus_count #total sentences

165

In [11]:
cbow_model.train(story_data, total_examples=cbow_model.corpus_count, epochs=cbow_model.epochs)

(27611, 107050)

In [12]:
print(cbow_model.wv["aman"])
print(len(cbow_model.wv["aman"]))

[ 1.14275618e-02  2.04915732e-01  1.41545897e-04  1.74545459e-02
 -3.22452411e-02 -1.13815390e-01  9.71569344e-02  3.30988348e-01
 -2.27073953e-02 -1.14119731e-01  7.11920038e-02 -3.14984508e-02
 -3.66857462e-02 -1.97561421e-02 -3.79682519e-02 -1.49728879e-01
  1.73500896e-01  6.45912364e-02 -3.46364267e-02 -7.80620798e-02
 -2.65409201e-02  1.00863064e-02  1.66119024e-01  1.07886106e-01
 -2.24714428e-02 -4.43077795e-02 -2.07604051e-01 -1.28355483e-02
 -4.18718755e-02 -1.66418388e-01  1.16050132e-01 -6.46317601e-02
  1.95185095e-02 -5.31108379e-02 -6.19626716e-02  1.15128905e-01
  5.06824441e-02 -1.36917248e-01 -9.02609080e-02  2.38736309e-02
 -3.05632781e-02  7.19523942e-03  4.54778001e-02 -1.38530254e-01
  8.87383595e-02  1.59861848e-01  5.06545082e-02  2.79643424e-02
  2.54738834e-02  5.07748015e-02  1.00830823e-01  1.96668645e-03
 -1.88957546e-02  6.64329752e-02  1.01070404e-02  5.49288355e-02
  5.44354208e-02 -2.39017345e-02  1.83854923e-02 -7.81865790e-03
 -8.60063359e-02 -8.56775

## Step 5: Inspect learned CBOW embeddings

- Retrieve a vector for a known word.
- Check vector length.
- Use `most_similar()` to inspect nearby words.
- Use `doesnt_match()` for outlier detection.
- Use `similarity()` for pairwise closeness.

These checks help verify whether the learned embedding space makes practical sense.

In [13]:
cbow_model.wv["sutradhar"]

array([ 0.01430061,  0.2248276 , -0.00194533,  0.01782533, -0.03142162,
       -0.12420379,  0.1058808 ,  0.35894707, -0.02441957, -0.12410109,
        0.07991103, -0.03441819, -0.04098203, -0.01700071, -0.04240347,
       -0.16259357,  0.19170594,  0.07138814, -0.04262741, -0.08074393,
       -0.0349678 ,  0.01784516,  0.18331318,  0.11883348, -0.02446575,
       -0.05199541, -0.22895886, -0.01341658, -0.04567176, -0.17929879,
        0.12707609, -0.07389428,  0.01626857, -0.05751736, -0.07120728,
        0.12651616,  0.05191907, -0.14803883, -0.09287915,  0.02061637,
       -0.03387761,  0.01018291,  0.04620406, -0.15476093,  0.09813995,
        0.17882827,  0.04866628,  0.03042783,  0.02845401,  0.05574091,
        0.11085588, -0.0014793 , -0.01858341,  0.06655181,  0.01508907,
        0.05973257,  0.06048298, -0.02257273,  0.02249988, -0.00934989,
       -0.1000308 , -0.09535155,  0.05817499,  0.04407108, -0.03431209,
        0.12951219,  0.0286567 ,  0.01960905, -0.15385848, -0.05

In [14]:
cbow_model.wv.most_similar("aman")

[('the', 0.9997098445892334),
 ('sutradhar', 0.9996960759162903),
 ('he', 0.9996883869171143),
 ('its', 0.9996796250343323),
 ('to', 0.9996777772903442),
 ('intelligence', 0.9996745586395264),
 ('and', 0.9996601343154907),
 ('more', 0.999660074710846),
 ('human', 0.9996565580368042),
 ('of', 0.9996530413627625)]

In [15]:
cbow_model.wv.doesnt_match(["aman", "sutradhar", "farmer", "queen", "boss", "nvidia"])

'nvidia'

In [16]:
cbow_model.wv.similarity("aman", "engineer")

0.99943584

In [17]:
skip_gram_model= Word2Vec(
    window=6,
    min_count=3,
    vector_size=300,
    epochs= 50,
    sg=1
)
print('hyperparameters set successfully!')

hyperparameters set successfully!


## Step 6: Train and compare Skip-Gram

- `sg=1` switches the model to Skip-Gram.
- Skip-Gram predicts surrounding words from a center word.
- It often performs better on smaller datasets and rarer words, but is slower than CBOW.

**Alternative:** FastText if subword handling is needed.

**Tradeoff:** CBOW is faster and stable; Skip-Gram can capture finer relationships on limited data.

In [18]:
skip_gram_model.build_vocab(story_data)
print("Vocabulary built successfully!")

Vocabulary built successfully!


In [19]:
skip_gram_model.train(story_data, total_examples=skip_gram_model.corpus_count, epochs=skip_gram_model.epochs)
print("Skip-gram model trained successfully!")

Skip-gram model trained successfully!


In [20]:
print(skip_gram_model.wv["aman"])
print(len(skip_gram_model.wv["aman"]))

[-3.69293010e-03  1.06090464e-01  1.31173506e-02  2.54328847e-02
 -1.68908853e-02 -5.45396581e-02  3.24940830e-02  2.10727364e-01
 -4.22812533e-03 -6.32644743e-02  3.40017118e-02 -3.92223783e-02
 -1.68203767e-02  7.25083519e-04 -3.33604366e-02 -7.97394142e-02
  9.71179307e-02  1.74544472e-02 -3.63961868e-02 -5.16330525e-02
 -2.71781497e-02  2.07981188e-03  9.86033976e-02  9.54427645e-02
 -1.88851338e-02 -9.53144766e-03 -1.26768857e-01 -8.49729870e-03
 -1.03702918e-02 -7.99073428e-02  7.95048326e-02 -4.27438617e-02
  1.21720079e-02 -2.13294588e-02 -4.26846221e-02  7.62200207e-02
  1.80024300e-02 -1.05206519e-01 -6.61209598e-02  1.73332561e-02
 -1.82599146e-02  1.69378251e-03  2.28906311e-02 -6.90321550e-02
  6.29786402e-02  1.10863961e-01  2.90100034e-02 -5.52508002e-03
  1.86976548e-02  2.11494956e-02  8.02196711e-02 -3.69818113e-03
 -3.59116658e-03  3.82306464e-02 -5.08206943e-03  3.17048132e-02
  3.31335627e-02 -2.92477999e-02 -1.72588937e-02 -7.51704071e-03
 -4.64716963e-02 -7.22756

In [21]:
skip_gram_model.wv.most_similar("aman")

[('over', 0.9986256957054138),
 ('but', 0.998593807220459),
 ('pratibimb', 0.998579740524292),
 ('stories', 0.9985631108283997),
 ('patterns', 0.9984599947929382),
 ('had', 0.9984554648399353),
 ('something', 0.9984305500984192),
 ('freedom', 0.9984069466590881),
 ('less', 0.998401403427124),
 ('risk', 0.9983949065208435)]